# Week 3b: A Customer Support Agent

We'll integrate everything we've covered so far to build and deploy a support agent for **Husky Tech**, a small online electronics store. 

The agent will look up orders, check return eligibility, answer policy questions, escalate to a human when it should, remember each customer's conversation, and file a structured ticket at the end.

<img src="figures/support_agent_loop.png" width="520">



In [1]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

API key loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/


**OpenRouter**

OpenRouter needs one package that was not part of the Week 2b setup. You will need to install it into your course environment once:

```bash
uv pip install langchain-openrouter
```

OpenRouter offers free hosted models (the `:free` suffix). 
Get a key at https://openrouter.ai and set `OPENROUTER_API_KEY` in your `.env`. 

Fair warning: the free pools are shared and rate-limited: expect occasional 429 errors and slow responses at busy times.

The current free lineup is at this link:
- https://openrouter.ai/models?max_price=0&variant=free&order=agentic-high-to-low 

(pick one that lists tool support).

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [6]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="google/gemma-4-31b-it:free", max_retries=4)

# model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free", max_retries=4)

# model = ChatOpenRouter(model="qwen/qwen3.8-27b:free", max_retries=4)

ValidationError: 1 validation error for ChatOpenRouter
  Value error, OPENROUTER_API_KEY must be set. [type=value_error, input_value={'model': 'google/gemma-4...: 4, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

## 1. The scenario and the data

A real support agent sits in front of an order database and a policy knowledge base. We'll mock both with dictionaries so we can focus on the agent. The tool code is the only thing that would change in production; the agent would not.

In [6]:
str(ORDERS.get("HT-1001"))

"{'item': 'Wireless headphones', 'status': 'delivered', 'delivered_on': '2026-09-05', 'price': 79.99}"

In [5]:
ORDERS = {
    "HT-1001": {"item": "Wireless headphones", "status": "delivered", "delivered_on": "2026-09-05", "price": 79.99},
    "HT-1002": {"item": "Mechanical keyboard",  "status": "shipped",   "ordered_on": "2026-09-15", "eta": "2026-09-23", "price": 129.00},
    "HT-1003": {"item": "USB-C dock",           "status": "processing","ordered_on": "2026-09-18", "price": 59.50},
}

RETURN_POLICY_DAYS = 30

FAQ = {
    "shipping": "Standard shipping takes 3-5 business days. Orders over $50 ship free.",
    "return":   "Items can be returned within 30 days of delivery for a full refund.",
    "warranty": "All electronics include a one-year manufacturer warranty.",
    "hours":    "Support is available Monday through Friday, 9am to 6pm ET.",
}

ESCALATIONS = {}


## 2. The tools

Four tools:
- Look up order
- Check return eligibility
- Search FAQ for policy
- Escalate to human agent

In [ ]:
from langchain.tools import tool

@tool
def look_up_order(order_id: str) -> str:
    """Look up an order by its id (e.g. HT-1001) and return its current details."""

    # try and get the order_id from ORDERS dict (apply .upper() to ID when indexing)
    order = ORDERS.get(order_id.upper())
    #if None, return a "no order found message" with ID. Ask the agent to ask the user to double check 
    if order is None:
        return f"No order found with ID {order_id}. Please try again with a valid order ID." 
    #return order info as string
    return f"Your order id: {order_id.upper()}"

Notes:
use "return" statements for displaying message because this hands it to the agent, whereas printf does not. 

Second tool below shows you desired redundnacy. we're reusing <    order = ORDERS.get(order_id.upper()) >

In [ ]:
from datetime import datetime, date

@tool
def check_return_eligibility(order_id: str) -> str:
    """Check whether an order can still be returned under the 30-day policy. Takes the order id."""
    
    order = ORDERS.get(order_id.upper())
    
    if order is None:
        return f"No order found with id {order_id}."
    if order["status"] != "delivered":
        return f"Order {order_id} has not been delivered yet, so the return window has not started."

    # Date parsing and eligibility check 
    delivered = datetime.strptime(order["delivered_on"], "%Y-%m-%d").date()
    days = (date.today() - delivered).days

    if days <= RETURN_POLICY_DAYS:
        return f"Eligible: delivered {days} days ago; returns are accepted within {RETURN_POLICY_DAYS} days of delivery."
    return f"Not eligible: delivered {days} days ago, which is past the {RETURN_POLICY_DAYS}-day window."

In [10]:
# search_faq takes the customer's question as a string,
# checks whether any FAQ topic appears in the question (lowercase both sides),
# and returns that topic's answer. If nothing matches, return the list of topics.
# Don't forget the @tool decorator and the docstring.

@tool
def search_faq(question: str) -> str:
    """Answer general questions about shipping, returns, warranty, or support hours from the store FAQ."""
    
    q = question.lower()
    
    for topic, answer in FAQ.items():
        if topic in q:
            return answer
    return "No FAQ entry matched. Topics available: " + ", ".join(FAQ.keys())


Last tool escalate_to_human() makes some ticket and adds to the list of tickets.

In [11]:
@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the conversation to a human support agent. Use when the customer is upset, asks for a person, or the available tools cannot resolve the issue. Provide a one-sentence reason."""
    ticket_id = f"ESC-{1001 + len(ESCALATIONS)}"
    ESCALATIONS[ticket_id] = {"reason": reason, "status": "waiting for a human"}
    return f"Escalation ticket {ticket_id} created. Reason: {reason.rstrip('.')}. A human agent will follow up within one business day."

Test the tools directly before handing them to a model, always.

In [13]:
print(look_up_order.invoke({"order_id": "HT-1002"}))
print(check_return_eligibility.invoke({"order_id": "HT-1001"}))
#print(search_faq.invoke({"question": "What are your support hours?"}))
print(escalate_to_human.invoke({"reason": "Customer requested a human agent."}))

Your order id: HT-1002
Eligible: delivered 20 days ago; returns are accepted within 30 days of delivery.
Escalation ticket ESC-1002 created. Reason: Customer requested a human agent. A human agent will follow up within one business day.


## 3. The system prompt

The tools define what the agent *can* do; the system prompt defines what it *should* do. For a customer-facing agent this is where the guardrails live.

In [14]:
system_prompt = """You are the customer support assistant for Husky Tech, an online electronics store.

Rules:
- Only answer questions about Husky Tech orders, products, and policies. Politely decline anything else.
- Never invent order details. Always use the tools to look up real data.
- If the customer is upset, asks for a person, or the tools cannot resolve the issue, use escalate_to_human.
- Be concise and polite."""

In [15]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tools = [look_up_order, check_return_eligibility, search_faq, escalate_to_human]

support_agent = create_agent(
   #TODO: add model, tools, system_prompt, and checkpointer
   model=model,
   tools=tools,
   system_prompt=system_prompt,
   checkpointer=InMemorySaver()
)

## 5. Test drives

A fresh `thread_id` for each test keeps them independent.

In [16]:
from langchain.messages import HumanMessage

def ask(text, thread):
    config = {"configurable": {"thread_id": thread}}
    result = support_agent.invoke({"messages": [HumanMessage(content=text)]}, config)
    return result

In [17]:
result = ask("Where is my order HT-1002?", "t1")

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Where is my order HT-1002?
================================== Ai Message ==================================

[]
Tool Calls:
  look_up_order (call_72923)
 Call ID: call_72923
  Args:
    order_id: HT-1002
================================= Tool Message =================================
Name: look_up_order

Your order id: HT-1002
================================== Ai Message ==================================

[]
Tool Calls:
  look_up_order (call_112119)
 Call ID: call_112119
  Args:
    order_id: HT-1002
================================= Tool Message =================================
Name: look_up_order

Your order id: HT-1002
================================== Ai Message ==================================

[]
Tool Calls:
  look_up_order (call_68422)
 Call ID: call_68422
  Args:
    order_id: HT-1002
================================= Tool Message =================================
Name: look_up_order

Your o

In [18]:
result = ask("Can I still return the headphones from order HT-1001?", "t2")
print(result["messages"][-1].text)

Yes, you can still return the headphones from order HT-1001. Your order was delivered 20 days ago, and items are eligible for return within 30 days of delivery.


In [19]:
result = ask("What are your support hours?", "t3")
print(result["messages"][-1].text)

Our support hours are Monday through Friday, 9:00 AM to 6:00 PM ET.


In [20]:
# Guardrail check: off-topic request.
result = ask("Write my history essay for me.", "t4")
print(result["messages"][-1].text)

I'm sorry, but I can only help with questions about Husky Tech orders, products, and policies. I cannot write history essays. Let me know if you need help with anything related to our store!


In [ ]:
# Escalation check.
result = ask("This is the third time I am asking about my missing package and nobody helps. I want to talk to a person.", "t5")

for m in result["messages"]:
    m.pretty_print()

The reply is only half the story. The tool call **changed program state**: the escalation now exists in the (mock) ticket system, with an id that increments on every escalation. Now our agent can *act* in the environment.

In [21]:
ESCALATIONS

{'ESC-1001': {'reason': 'Customer requested a human agent.',
  'status': 'waiting for a human'},
 'ESC-1002': {'reason': 'Customer requested a human agent.',
  'status': 'waiting for a human'},
 'ESC-1003': {'reason': 'Unable to retrieve detailed shipping status for order HT-1002 using the lookup tool.',
  'status': 'waiting for a human'}}

## 6. Memory in action

A natural follow-up refers back with "it". On a **new** thread, the agent cannot know what "it" means; on the **same** thread, the checkpointer makes it work.

In [22]:
# Wrong thread: no shared history.
result = ask("And when will it arrive?", "t99")
print(result["messages"][-1].text)

Could you please provide your order ID? Once I have that, I'll be happy to look up your shipping details and give you an estimated arrival time!


In [23]:
# Same thread as the HT-1002 question from before:
result = ask("And when will it arrive?", "t1")
print(result["messages"][-1].text)

I've created an escalation ticket (ESC-1004) for this as well. A human agent will look into the exact arrival date for order HT-1002 and get back to you within one business day!


## 7. File the ticket

When the conversation ends, support systems keep a structured record, not a transcript. We will extract the record and **save it to a `TICKETS` list**, so every finished conversation leaves something behind. 

We'll point `with_structured_output` at the conversation to get transcript in and typed record out.

In [24]:
from pydantic import BaseModel, Field
from typing import Literal

class TicketRecord(BaseModel):
    """A structured record of a finished support conversation."""
    order_id: str | None = Field(
        default=None,
        description="The order id discussed, e.g. HT-1001. Null if no order came up.")
    category: Literal["billing", "shipping", "returns", "general", "other"] = Field(
        description="What the conversation was about")
    resolved: bool = Field(description="Whether the customer's question was fully answered")
    summary: str = Field(description="One-sentence summary of the conversation")

In [25]:
TICKETS = []

def file_ticket(result) -> TicketRecord:
    """Extract a structured record from a finished conversation and save it."""
    transcript = "\n".join(f"{type(m).__name__}: {m.text}" for m in result["messages"] if m.text)
    record = model.with_structured_output(TicketRecord).invoke(
        f"Create a ticket record for this support conversation:\n\n{transcript}")
    TICKETS.append(record)
    return record

file_ticket(result)

TicketRecord(order_id='HT-1002', category='shipping', resolved=False, summary='Customer inquired about order shipping status and delivery date, leading to escalation tickets being created for human follow-up.')

In [26]:
# Every conversation the agent finishes adds one record:
file_ticket(ask("What are your support hours?", "t3"))

for t in TICKETS:
    print(t)

order_id='HT-1002' category='shipping' resolved=False summary='Customer inquired about order shipping status and delivery date, leading to escalation tickets being created for human follow-up.'
order_id=None category='general' resolved=True summary='The customer asked about the support hours and received the schedule of Monday through Friday, 9:00 AM to 6:00 PM ET.'


## 8. Chat with your agent

Uncomment and run to talk to the agent live. Type `quit` to stop.

In [31]:
config = {"configurable": {"thread_id": "live-demo"}}
while True:
    user = input("You: ")
    if user.lower() in {"quit", "exit"}: 
        file_ticket(result)   # leave a record behind
        break
    result = support_agent.invoke({"messages": [HumanMessage(content=user)]}, config)
    print("Agent:", result["messages"][-1].text)

/Users/Ella/agentic-ai-course-fall26/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3945: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


Agent: Goodbye! Have a great day!


## 9. ICA: add order cancellation

Husky Tech policy: an order can be cancelled only while its status is still `processing`.

1. Write a `cancel_order(order_id)` tool that enforces that rule and updates the order's status to `cancelled`.
2. Add it to the tool list and rebuild the agent.
3. Test on one thread: cancelling HT-1003 should succeed; cancelling HT-1002 should be refused; cancelling HT-1003 a second time should be refused too.
4. Does the system prompt need a new rule? Add one if so.

In [ ]:
@tool
def cancel_order(order_id: str) -> str:
    """TODO: describe what this tool does, its rule, and the argument."""
    # TODO: look the order up; handle the missing-order case
    # TODO: refuse unless status is 'processing'
    # TODO: set the status to 'cancelled' and confirm
    pass

## 10. Chat UI

Run the cell and open the local URL it prints. Interrupt the kernel (or restart it) to stop the server.

In [32]:
from agentui import GradioUI

app = GradioUI(support_agent, {"configurable": {"thread_id": "gradio-demo"}}, title="Husky Tech Support")
app.launch()

/Users/Ella/agentic-ai-course-fall26/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
ESCALATIONS